# ChatQEC on real hardware: IBM Quantum and IQM

ChatQEC can talk to real quantum hardware in plain English. For **IBM Quantum** and the **ORNL IQM** machine (20 qubits, native `r` and `cz` gates) it can:

* list the backends you can use and whether they are up,
* read live calibration data (T1, T2, readout error, two-qubit gate error) and always say how old it is,
* map a circuit onto the device with a quality-aware placer, or with **mapomatic**, which compiles first and scores the routed circuit,
* preview a run for free, and submit one only after you confirm (capped at 10 qubits and 4000 shots).

Each cell below sends one question to the `chatqec ask` CLI, the same pipeline the Streamlit app uses, and prints the answer together with the tool calls it made. Nothing in this notebook submits a hardware job unless you set `RUN_HARDWARE = True` in the last section.

## Setup

Needs `QISKIT_IBM_TOKEN`, `IQM_TOKEN` and `IQM_SERVER_URL` in `.env`, the `chatqec-mcp-tools` server installed, the IQM environment (`bash ~/chatqec-mcp-tools/scripts/setup_iqm_env.sh`), and Qdrant running. `PROVIDER = "local"` uses the local Muse-Glimmer model on port 8001; set it to `None` to use the configured API provider.

In [ ]:
import os
import re
import subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHATQEC = ROOT / ".venv" / "bin" / "chatqec"
PROVIDER = "local"  # None -> the provider in conf/config.yaml
LOCAL_MODEL = "muse-glimmer"

# CLI noise that says nothing about the answer
_NOISE = re.compile(
    r"UserWarning|show_warning|Loading weights|Hugging Face Hub refused"
    r"|qiskit_runtime_service|^\[chatqec\] backend="
)


def ask(question: str, timeout: int = 600) -> str:
    """Ask ChatQEC one question through the CLI, with the hardware tools enabled."""
    env = dict(os.environ)
    if PROVIDER:
        env.update(CHATQEC_PROVIDER=PROVIDER, CHATQEC_LOCAL_MODEL=LOCAL_MODEL)
    proc = subprocess.run(
        [str(CHATQEC), "ask", "--mcp-server", "chatqec-mcp-tools", question],
        cwd=ROOT,
        env=env,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )
    lines = (proc.stdout + proc.stderr).splitlines()
    text = "\n".join(line for line in lines if not _NOISE.search(line))
    print(f"Q: {question}\n\n{text.strip()}")
    return text

## IBM Quantum

### 1. Which backends can I use?
`ibm_list_backends`: name, qubit count, operational status and queue depth.

In [ ]:
_ = ask("What IBM quantum backends do I have access to, and which one has the shortest queue?")

### 2. Live calibration
`ibm_backend_calibration`: per-qubit T1, T2 and readout error, per-coupler two-qubit gate error. The answer states the data's age.

In [ ]:
_ = ask("On ibm_fez, which five qubits have the longest T1, and what are their readout errors?")

### 3. Map a circuit onto a backend
`ibm_map_circuit`: physical qubits, placement strategy, estimated fidelity and a coupling-map diagram.

In [ ]:
_ = ask(
    "Map a Bell state circuit onto the least busy IBM backend. Which physical qubits does it use?"
)

### 4. Default mapper vs mapomatic
The default mapper places the logical circuit directly. mapomatic compiles the circuit first, then picks the lowest-error placement of the routed circuit, counting the gates that SWAPs add. On heavy-hex chips (no qubit has four neighbours) the two can differ a lot for a GHZ star.

In [ ]:
_ = ask(
    "Map a 5-qubit GHZ circuit onto ibm_fez with mapomatic and compare it with the default mapper."
)

### 5. Preview a run (free, nothing is submitted)
`ibm_preview_execution`: mapped qubits, estimated fidelity, backend status and queue position.

In [ ]:
_ = ask("Preview running a 4-qubit QFT on ibm_fez with 1000 shots. Do not submit it.")

### 6. Error detection with Pauli checks
`ibm_add_pauli_checks` (IBM's qiskit-paulice): adds spacetime Pauli checks to a Clifford circuit and reports the overhead and the postselection cost.

In [ ]:
_ = ask(
    "Add two Pauli checks to this GHZ circuit and tell me the overhead: "
    'OPENQASM 2.0; include "qelib1.inc"; qreg q[3]; creg c[3]; '
    "h q[0]; cx q[0],q[1]; cx q[1],q[2]; measure q -> c;"
)

### 7. Fault-tolerant compilation
`ibm_ft_compile`: Clifford+T synthesis with gridsynth and the resulting T count.

In [ ]:
_ = ask(
    "Compile this circuit to Clifford+T and report the T count: "
    'OPENQASM 2.0; include "qelib1.inc"; qreg q[2]; h q[0]; rz(0.3) q[0]; cx q[0],q[1]; rz(0.7) q[1];'
)

## IQM (ORNL)

The IQM tools run the IQM SDK in its own environment and reuse the same placement engines as IBM. Qubits are named `QB1` to `QB20`. The server reports device health but not queue depth.

### 1. What machine is behind the server?
`iqm_list_backends`: chip label, qubits, couplers, native gates, calibration set and health.

In [ ]:
_ = ask("What IQM quantum computer do I have access to, and is it healthy right now?")

### 2. Calibration and the best qubits
`iqm_backend_calibration`: T1, T2 echo, T2*, readout fidelity, single-qubit RB fidelity, and per-coupler CZ fidelity (interleaved RB) and duration.

In [ ]:
_ = ask("Show me the T1 and CZ fidelity of the best qubits on the IQM machine.")

### 3. Which couplers are worst?
Useful for knowing which part of the chip to avoid.

In [ ]:
_ = ask(
    "Which three couplers on the IQM machine have the lowest CZ fidelity, and which qubits do they connect?"
)

### 4. Map a GHZ circuit
`iqm_map_circuit`: physical qubits as QB names, the logical-to-physical map, estimated fidelity and the coupling-map diagram.

In [ ]:
_ = ask("Map a 4-qubit GHZ circuit onto the IQM hardware and show which physical qubits it uses.")

### 5. mapomatic on IQM
For a QFT the routed circuit needs extra CZs; mapomatic counts them, so its fidelity estimate is the more realistic one.

In [ ]:
_ = ask(
    "Map a 4-qubit QFT onto the IQM hardware with mapomatic and with the default mapper, and compare the two."
)

### 6. Preview a run (free)
`iqm_preview_execution`: everything a submission would use, without submitting.

In [ ]:
_ = ask("Map a 4-qubit GHZ circuit onto the IQM hardware and preview running it with 500 shots.")

### 7. What happens before a job is submitted?
The assistant explains the safety flow: preview first, submit only on explicit confirmation, then fetch counts by job id.

In [ ]:
_ = ask(
    "I want to run a Bell circuit on the IQM quantum computer. What happens before anything is submitted, and how do I get the counts afterwards?"
)

### 8. Optional: a real run (uses your IQM allocation)
Set `RUN_HARDWARE = True` to submit a 100-shot Bell circuit, then fetch its counts with `iqm_job_result`. A queued job may need the second cell re-run after a while.

In [ ]:
RUN_HARDWARE = False
JOB_ID = None
if RUN_HARDWARE:
    out = ask("Yes, I confirm: run a Bell circuit on the IQM quantum computer with 100 shots.")
    m = re.search(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", out)
    JOB_ID = m.group(0) if m else None
    print("\njob id:", JOB_ID)

In [ ]:
if JOB_ID:
    _ = ask(f"What is the status of IQM job {JOB_ID}, and what are its measurement counts?")